# Disaster Tweet Classification: GloVe Pretrained Recurrent Benchmark

**Embedding:** GloVe (Global Vectors for Word Representation, Twitter / Wiki-Gigaword 100-dim)  
**Models:** GloVe + BiRNN, GloVe + BiLSTM, GloVe + 2-Stacked BiLSTM  
**Objective:** Leverage global co-occurrence statistics from pretrained GloVe vectors, fine-tune recurrent feature extractors, and evaluate class-level performance across 10 humanitarian disaster categories.

---
### Notebook Structure
1. **Dataset Ingestion & Environment Setup**
2. **GloVe Pretrained Vector Loading & Vocabulary Construction**
3. **PyTorch Recurrent Model Definitions (BiRNN, BiLSTM, Stacked BiLSTM)**
4. **Training Engine (Class-Weighted CrossEntropy, ReduceLROnPlateau, Early Stopping)**
5. **Model Checkpointing & Comparative Evaluation**
6. **Visualization Suite (Training Curves, Confusion Matrices, Per-Class Bar Charts, Imbalance Robustness)**

In [ ]:
import os
import sys
import time
import random
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
import gensim.downloader as api
from gensim.models import KeyedVectors

# Setup directories
DATA_DIR = Path("dataset")
RESULTS_DIR = Path("results/04_glove_recurrent_models")
MODELS_DIR = RESULTS_DIR / "saved_models"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Device & Reproducibility
SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[+] Utilizing compute device: {device}")

train_path = DATA_DIR / "train_clean.parquet"
val_path = DATA_DIR / "validation_clean.parquet"
test_path = DATA_DIR / "test_clean.parquet"

# Sourcing dataset from Kaggle or Google Drive
KAGGLE_INPUT_DIR = Path("/kaggle/input/humaid-disaster-tweets-parquet")
if KAGGLE_INPUT_DIR.exists():
    print("[+] Sourcing dataset from Kaggle dataset input...")
    for split in ["train", "validation", "test"]:
        p_clean = KAGGLE_INPUT_DIR / f"{split}_clean.parquet"
        p_raw = KAGGLE_INPUT_DIR / f"{split}.parquet"
        target_p = DATA_DIR / f"{split}_clean.parquet"
        if not target_p.exists():
            if p_clean.exists():
                pd.read_parquet(p_clean).to_parquet(target_p)
            elif p_raw.exists():
                pd.read_parquet(p_raw).to_parquet(target_p)

if not (train_path.exists() and val_path.exists() and test_path.exists()):
    raw_train = DATA_DIR / "train.parquet"
    raw_val = DATA_DIR / "validation.parquet"
    raw_test = DATA_DIR / "test.parquet"
    if not (raw_train.exists() and raw_val.exists() and raw_test.exists()):
        print("[+] Downloading HumAID dataset from Google Drive...")
        import gdown
        GDRIVE_URL = "https://drive.google.com/drive/folders/1pyMBc4SFc-sQvfmReiywPoN5cQbMpQBR?usp=drive_link"
        gdown.download_folder(url=GDRIVE_URL, output=str(DATA_DIR), quiet=False, use_cookies=False)

train_file = train_path if train_path.exists() else DATA_DIR / "train.parquet"
val_file = val_path if val_path.exists() else DATA_DIR / "validation.parquet"
test_file = test_path if test_path.exists() else DATA_DIR / "test.parquet"

train_df = pd.read_parquet(train_file)
val_df = pd.read_parquet(val_file)
test_df = pd.read_parquet(test_file)

text_col = "clean_text" if "clean_text" in train_df.columns else "tweet_text"
print(f"[+] Loaded splits using column '{text_col}':")
print(f"    Train: {len(train_df):,} samples | Val: {len(val_df):,} samples | Test: {len(test_df):,} samples")

## 1. GloVe Vector Loading & Vocabulary Construction
We load pretrained GloVe vectors (`glove-twitter-100` / `glove-wiki-gigaword-100`) via Gensim Downloader with offline fallback.

In [ ]:
# Tokenize cleaned text
train_tokens = [str(t).split() for t in train_df[text_col]]
val_tokens = [str(t).split() for t in val_df[text_col]]
test_tokens = [str(t).split() for t in test_df[text_col]]

# Encode labels
class_names = sorted(train_df["class_label"].unique())
label2idx = {name: i for i, name in enumerate(class_names)}
idx2label = {i: name for i, name in enumerate(class_names)}

y_train = train_df["class_label"].map(label2idx).values
y_val = val_df["class_label"].map(label2idx).values
y_test = test_df["class_label"].map(label2idx).values

# Load GloVe Pretrained Embeddings
EMBEDDING_DIM = 100
print("[+] Loading GloVe Pretrained Vectors...")
try:
    glove_vectors = api.load("glove-twitter-100")
    print("[+] Successfully loaded glove-twitter-100.")
except Exception as e:
    print(f"[-] Twitter glove failed ({{e}}), attempting glove-wiki-gigaword-100...")
    try:
        glove_vectors = api.load("glove-wiki-gigaword-100")
    except Exception as e2:
        print("[-] Fallback: Training domain GloVe/Word2Vec 100d...")
        from gensim.models import Word2Vec
        w2v_fb = Word2Vec(sentences=train_tokens, vector_size=EMBEDDING_DIM, window=5, min_count=2, sg=1, epochs=10)
        glove_vectors = w2v_fb.wv

# Build Vocabulary
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}

# Add train tokens to vocab
for tokens in train_tokens:
    for word in tokens:
        if word not in vocab:
            vocab[word] = len(vocab)

VOCAB_SIZE = len(vocab)
print(f"[+] Vocabulary built with {VOCAB_SIZE:,} unique tokens.")

# Construct Embedding Matrix
embedding_matrix = np.zeros((VOCAB_SIZE, EMBEDDING_DIM), dtype=np.float32)
embedding_matrix[1] = np.random.normal(scale=0.6, size=(EMBEDDING_DIM,)) # UNK token
matched_words = 0
for word, idx in vocab.items():
    if word in glove_vectors:
        embedding_matrix[idx] = glove_vectors[word]
        matched_words += 1

print(f"[+] GloVe matched {matched_words:,} / {VOCAB_SIZE:,} tokens ({matched_words/VOCAB_SIZE*100:.2f}% coverage).")

# Dataset & DataLoader
MAX_LEN = 64
def text_to_indices(tokens, max_len=MAX_LEN):
    indices = [vocab.get(w, vocab[UNK_TOKEN]) for w in tokens[:max_len]]
    if len(indices) < max_len:
        indices += [vocab[PAD_TOKEN]] * (max_len - len(indices))
    return indices

class TweetDataset(Dataset):
    def __init__(self, tokenized_texts, labels):
        self.X = torch.tensor([text_to_indices(t) for t in tokenized_texts], dtype=torch.long)
        self.y = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TweetDataset(train_tokens, y_train)
val_dataset = TweetDataset(val_tokens, y_val)
test_dataset = TweetDataset(test_tokens, y_test)

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Compute Class Weights for Loss Function
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
print(f"[+] Initialized weighted CrossEntropyLoss across {len(class_names)} classes.")

## 2. GloVe Recurrent Model Architectures & Training Engine

In [ ]:
class GloVeRecurrentClassifier(nn.Module):
    def __init__(self, embedding_matrix, cell_type="bilstm", hidden_dim=128, num_layers=1, num_classes=10, dropout=0.3):
        super(GloVeRecurrentClassifier, self).__init__()
        vocab_size, emb_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix, dtype=torch.float32), freeze=False)
        self.cell_type = cell_type.lower()
        self.bidirectional = "bi" in self.cell_type
        
        rnn_dropout = dropout if num_layers > 1 else 0.0
        
        if "lstm" in self.cell_type:
            self.rnn = nn.LSTM(
                emb_dim, hidden_dim, num_layers=num_layers,
                bidirectional=self.bidirectional, batch_first=True, dropout=rnn_dropout
            )
        else:
            self.rnn = nn.RNN(
                emb_dim, hidden_dim, num_layers=num_layers,
                bidirectional=self.bidirectional, batch_first=True, dropout=rnn_dropout, nonlinearity='tanh'
            )
            
        fc_in = hidden_dim * 2 if self.bidirectional else hidden_dim
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(fc_in, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        embedded = self.embedding(x)
        if "lstm" in self.cell_type:
            out, (hn, cn) = self.rnn(embedded)
        else:
            out, hn = self.rnn(embedded)
            
        pooled, _ = torch.max(out, dim=1)
        pooled = self.dropout(pooled)
        logits = self.fc(pooled)
        return logits

def train_and_evaluate_model(model_name, cell_type, num_layers, epochs=12, lr=1e-3):
    print(f"\n=======================================================")
    print(f"[+] Training Model: {model_name} (cell={cell_type}, layers={num_layers})")
    print(f"=======================================================")
    
    model = GloVeRecurrentClassifier(
        embedding_matrix=embedding_matrix,
        cell_type=cell_type,
        hidden_dim=128,
        num_layers=num_layers,
        num_classes=len(class_names),
        dropout=0.3
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    
    history = {"train_loss": [], "val_loss": [], "val_macro_f1": []}
    best_val_f1 = 0.0
    best_weights_path = MODELS_DIR / f"{model_name.lower().replace(' ', '_')}.pt"
    
    for epoch in range(1, epochs + 1):
        # Training Phase
        model.train()
        total_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            total_loss += loss.item() * len(y_batch)
            
        train_loss = total_loss / len(train_dataset)
        
        # Validation Phase
        model.eval()
        val_loss, all_preds, all_labels = 0.0, [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                logits = model(X_batch)
                loss = criterion(logits, y_batch)
                val_loss += loss.item() * len(y_batch)
                preds = torch.argmax(logits, dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(y_batch.cpu().numpy())
                
        val_loss /= len(val_dataset)
        val_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        scheduler.step(val_f1)
        
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_macro_f1"].append(val_f1)
        
        print(f"    Epoch {epoch:02d}/{epochs:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro F1: {val_f1:.4f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_weights_path)
            
    # Load Best Model for Final Test Evaluation
    model.load_state_dict(torch.load(best_weights_path))
    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            test_preds.extend(preds)
            test_labels.extend(y_batch.numpy())
            
    test_acc = accuracy_score(test_labels, test_preds)
    test_f1_macro = f1_score(test_labels, test_preds, average='macro', zero_division=0)
    test_f1_weighted = f1_score(test_labels, test_preds, average='weighted', zero_division=0)
    test_prec_macro = precision_score(test_labels, test_preds, average='macro', zero_division=0)
    test_rec_macro = recall_score(test_labels, test_preds, average='macro', zero_division=0)
    
    print(f"[+] Final Test Results for {model_name}:")
    print(f"    --> Macro F1: {test_f1_macro:.4f} | Accuracy: {test_acc:.4f} | Weighted F1: {test_f1_weighted:.4f}")
    
    return {
        "model_name": model_name,
        "history": history,
        "test_preds": np.array(test_preds),
        "test_acc": test_acc,
        "test_f1_macro": test_f1_macro,
        "test_f1_weighted": test_f1_weighted,
        "test_prec_macro": test_prec_macro,
        "test_rec_macro": test_rec_macro
    }

## 3. Train GloVe Recurrent Models (BiRNN, BiLSTM, 2-Stacked BiLSTM)

In [ ]:
experiments = [
    {"name": "GloVe + BiRNN", "cell_type": "birnn", "num_layers": 1, "epochs": 10},
    {"name": "GloVe + BiLSTM", "cell_type": "bilstm", "num_layers": 1, "epochs": 10},
    {"name": "GloVe + 2-Stacked BiLSTM", "cell_type": "bilstm", "num_layers": 2, "epochs": 10},
]

experiment_results = {}
summary_metrics = []

for exp in experiments:
    res = train_and_evaluate_model(
        model_name=exp["name"],
        cell_type=exp["cell_type"],
        num_layers=exp["num_layers"],
        epochs=exp["epochs"]
    )
    experiment_results[exp["name"]] = res
    summary_metrics.append({
        "Model": exp["name"],
        "Cell Type": exp["cell_type"].upper(),
        "Layers": exp["num_layers"],
        "Test Accuracy": res["test_acc"],
        "Test Macro F1": res["test_f1_macro"],
        "Test Weighted F1": res["test_f1_weighted"],
        "Test Macro Precision": res["test_prec_macro"],
        "Test Macro Recall": res["test_rec_macro"],
    })

summary_df = pd.DataFrame(summary_metrics).sort_values(by="Test Macro F1", ascending=False)
summary_df.to_csv(RESULTS_DIR / "metrics_comparison.csv", index=False)
print("\n=== GloVe Recurrent Architectures Benchmark Summary ===")
print(summary_df.to_string(index=False))

## 4. Visualizations & Diagnostic Reports

In [ ]:
# 1. Unified Training Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6), dpi=120)
for name, res in experiment_results.items():
    ax1.plot(res["history"]["val_loss"], label=name, linewidth=2)
    ax2.plot(res["history"]["val_macro_f1"], label=name, linewidth=2)

ax1.set_title("Validation Loss Progression", fontsize=12, fontweight='bold')
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend(loc="upper right")

ax2.set_title("Validation Macro F1-Score Progression", fontsize=12, fontweight='bold')
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Macro F1")
ax2.legend(loc="lower right")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_curves.png", bbox_inches='tight')
plt.show()

# 2. Detailed Diagnostic Reports for Best GloVe Model
best_model_name = summary_df.iloc[0]["Model"]
best_preds = experiment_results[best_model_name]["test_preds"]
print(f"[+] Generating Diagnostic Visualizations for Top Performer: {best_model_name}")

# Classification Report
report_str = classification_report(y_test, best_preds, target_names=class_names, digits=4)
with open(RESULTS_DIR / "classification_report.txt", "w", encoding="utf-8") as f:
    f.write(f"=== Classification Report: {best_model_name} ===\n\n")
    f.write(report_str)
print(report_str)

# Dual Confusion Matrices
cm_raw = confusion_matrix(y_test, best_preds)
cm_norm = cm_raw.astype('float') / cm_raw.sum(axis=1)[:, np.newaxis]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8), dpi=120)
sns.heatmap(cm_raw, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax1)
ax1.set_title(f"Raw Confusion Matrix: {best_model_name}", fontsize=12, fontweight='bold')
ax1.set_xlabel("Predicted Label")
ax1.set_ylabel("True Label")
ax1.tick_params(axis='x', rotation=45)

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax2)
ax2.set_title(f"Normalized Confusion Matrix: {best_model_name}", fontsize=12, fontweight='bold')
ax2.set_xlabel("Predicted Label")
ax2.set_ylabel("True Label")
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrices.png", bbox_inches='tight')
plt.show()

# Per-Class Metrics Bar Chart
report_dict = classification_report(y_test, best_preds, target_names=class_names, output_dict=True)
per_class_df = pd.DataFrame([
    {
        "Class": cls,
        "Precision": report_dict[cls]["precision"],
        "Recall": report_dict[cls]["recall"],
        "F1-Score": report_dict[cls]["f1-score"],
        "Support": report_dict[cls]["support"]
    }
    for cls in class_names
])
per_class_df.to_csv(RESULTS_DIR / "per_class_metrics.csv", index=False)

fig, ax = plt.subplots(figsize=(14, 7), dpi=120)
x = np.arange(len(class_names))
width = 0.25

ax.barh(x - width, per_class_df["Precision"], width, label="Precision", color="#3498db")
ax.barh(x, per_class_df["Recall"], width, label="Recall", color="#2ecc71")
ax.barh(x + width, per_class_df["F1-Score"], width, label="F1-Score", color="#e74c3c")

ax.set_yticks(x)
ax.set_yticklabels(class_names, fontsize=10)
ax.set_xlabel("Score", fontsize=11, fontweight='bold')
ax.set_title(f"Per-Class Performance Metrics: {best_model_name}", fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim(0, 1.05)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "per_class_metrics.png", bbox_inches='tight')
plt.show()

# Imbalance Robustness Correlation
fig, ax = plt.subplots(figsize=(9, 5), dpi=120)
sns.regplot(data=per_class_df, x="Support", y="F1-Score", scatter_kws={'s': 60, 'color': '#8e44ad'}, line_kws={'color': '#2980b9'}, ax=ax)
ax.set_title(f"Class Support vs. F1-Score: {best_model_name}", fontsize=12, fontweight='bold')
ax.set_xlabel("Test Class Support (Number of Samples)", fontsize=11)
ax.set_ylabel("Class F1-Score", fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "imbalance_robustness_correlation.png", bbox_inches='tight')
plt.show()

print("[+] All artifacts and plots successfully saved to:", RESULTS_DIR.resolve())